## 05 - FrontEnd Architecture

Purpose: build realistic frontend and test / optimize

In [ ]:
# Imports

import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
import soundfile as sf
import time

from util import sigmoid_squash
from util import MAX_BW, MIN_BW, MAX_DEGREE, MIN_DEGREE, MAX_DUR_MS, MIN_DUR_MS, MIN_RATIO, MIN_FREQ, MAX_FREQ, FS

%matplotlib widget

In [ ]:
# Define frontend class
#    Note that learnable parameters theta_<param> get mapped to the actual inputs that create chirplets after passing though softplus/sigmoid for continuity

class ChirpletFilterbank(nn.Module):

    def __init__(self, n_channels):
        super().__init__()

        # Parameters
        self.n_channels = n_channels

        # Parameter constraints
        self.MAX_BW = MAX_BW        
        self.MIN_BW = MIN_BW
        self.MAX_DEGREE = MAX_DEGREE
        self.MIN_DEGREE = MIN_DEGREE
        self.MAX_DUR_MS = MAX_DUR_MS
        self.MIN_DUR_MS = MIN_DUR_MS
        self.MIN_RATIO = MIN_RATIO
        self.MAX_FREQ = MAX_FREQ
        self.MIN_FREQ = MIN_FREQ

        # Learnable parameter registration
        self.theta_bw = nn.Parameter(torch.zeros(n_channels))
        self.theta_fc = nn.Parameter(torch.zeros(n_channels))
        self.theta_T = nn.Parameter(torch.zeros(n_channels))
        self.theta_degree = nn.Parameter(torch.zeros(n_channels))
        self.theta_sign = nn.Parameter(torch.zeros(n_channels))
        

    def _get_constrained_params(self):
        """
        Apply warping so that continuous theta_* parameters respect constraints before being fed as chirplet gen inputs

        Take self.theta_* and return constrained version

        Internal method, called only by forward()
        """

        bw = sigmoid_squash(self.theta_bw, self.MIN_BW, self.MAX_BW)

        # fc's valid range depends on bw
        fc_lo = self.MIN_FREQ + bw / 2
        fc_hi = torch.min(self.MAX_FREQ - bw / 2, bw * (self.MIN_RATIO + 1) / (2 * (self.MIN_RATIO - 1)))
        fc = sigmoid_squash(self.theta_fc, fc_lo, fc_hi)

        T = sigmoid_squash(self.theta_T, self.MIN_DUR_MS, self.MAX_DUR_MS)
        degree = sigmoid_squash(self.theta_degree, self.MIN_DEGREE, self.MAX_DEGREE)

        return (bw, fc, T, degree)
                               
    def _generate_kernels(self, bw, fc, T, degree):
        """
        Generate chirplet kernels from parameter tensors
        """

        # Compute f1, f2 from bw and fc tensors
        f1 = fc - bw / 2
        f2 = fc + bw / 2

        # Incorporate sweep direction
        w = torch.sigmoid(self.theta_sign)
        snap = (w > 0.5).float().detach() # snaps to 0 or 1 but removes grad flow
        disc = snap + (w - w.detach()) # allows for w gradient flow in backprop
        f_start = disc * f1 + (1 - disc) * f2
        f_end = disc * f2 + (1 - disc) * f1

        # Time vector (u = t / T)
        Tmax = self.MAX_DUR_MS / 1000.
        t = torch.arange(int(Tmax * self.FS), device=bw.device)/ self.FS # keep this tensor on same device as others
        t_b = t.unsqueeze(0) # for broadcasting
        T_b = T.unsqueeze(1) # for broadcasting
        mask = (t_b < T_b).float() # get valid range based on variable T per chan
        u =  t_b / T_b # normalized time arg 

        # Inst freq
        fsb = f_start.unsqueeze(1)
        feb = f_end.unsqueeze(1)
        degb = degree.unsqueeze(1)
        f_i = (fsb + (feb - fsb) * u ** degb) * mask

        # Inst phase - cumulative trapezoidal integration
        dt = 1.0 / self.FS
        phi_i = 2 * np.pi * dt * torch.cumsum((f_i[:, :-1] + f_i[:, 1:]) / 2, dim=1)
        phi_i = F.pad(phi_i, (1, 0))  # prepend 0, matching cumulative_trapezoid's initial=0
        
        # Variable length hann window per channel
        win = 0.5 * (1. - torch.cos(2 * np.pi * u)) * mask

        return torch.sin(phi_i) * win

    def forward(self, x):
        """
        Forward propagate

        x: [batch, 1, signal_len]
        """

        # Get kernel gen params
        bw, fc, T, degree = self._get_constrained_params()

        # Create kernels, reshape, and time reverse for actual conv
        k = self._generate_kernels(bw, fc, T, degree) # out: [n_channels, kernel_len]
        k = k.unsqueeze(1) # conv1d expects [out_ch, in_ch, kernel_len]
        k = k.flip(-1)

        # Conv
        y = F.conv1d(x, k, padding='same')
        
        